# Imitation learning : apprendre a partir de demonstrations expertes

Ce notebook reproduit le mecanisme central du **Home Task 2 officiel IOAI 2026** ("Robot Delivery Academy") : un robot doit apprendre a livrer un colis sur une grille en observant des trajectoires expertes, sans qu'on lui donne les regles explicitement. La technique s'appelle le **behavioral cloning** (BC) : on transforme le probleme en une classification supervisee classique ou l'entree est l'observation et la sortie est l'action que l'expert a prise.

Reference officielle : https://github.com/IOAI-official/IOAI-2026 (dossier `Home Task`, tache 2, grille 8x8, 6 depots).

Ici on simplifie a une grille plus petite pour que tout tourne en quelques secondes, mais le principe est identique.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

## 1. Le monde : une grille avec un depart et une arrivee

Actions possibles : haut, bas, gauche, droite (0, 1, 2, 3). L'"expert" ici est une regle simple qui bouge toujours vers la cible (on pourrait le remplacer par un vrai planificateur, l'idee ne change pas).

In [ ]:
TAILLE_GRILLE = 6
ACTIONS = ["haut", "bas", "gauche", "droite"]

def politique_experte(pos, cible):
    """Regle simple : reduit d'abord l'ecart vertical, puis horizontal."""
    ly, lx = pos
    cy, cx = cible
    if ly != cy:
        return 0 if cy < ly else 1   # haut ou bas
    if lx != cx:
        return 2 if cx < lx else 3   # gauche ou droite
    return None   # deja arrive

def appliquer_action(pos, action):
    y, x = pos
    if action == 0: y -= 1
    elif action == 1: y += 1
    elif action == 2: x -= 1
    elif action == 3: x += 1
    y = max(0, min(TAILLE_GRILLE - 1, y))
    x = max(0, min(TAILLE_GRILLE - 1, x))
    return (y, x)

def observation(pos, cible):
    """On encode position + cible en vecteur normalise, comme le 'vector' du Home Task officiel."""
    y, x = pos
    cy, cx = cible
    return np.array([y, x, cy, cx, cy - y, cx - x], dtype=np.float32) / TAILLE_GRILLE

## 2. Collecter des demonstrations expertes

C'est l'equivalent du `train_demos.pkl` fourni dans le Home Task officiel. On genere plusieurs episodes complets en suivant la politique experte, et on enregistre chaque paire (observation, action).

In [ ]:
def generer_episode(graine):
    rng = np.random.RandomState(graine)
    pos = (rng.randint(TAILLE_GRILLE), rng.randint(TAILLE_GRILLE))
    cible = (rng.randint(TAILLE_GRILLE), rng.randint(TAILLE_GRILLE))
    obs_liste, act_liste = [], []
    for _ in range(20):   # limite de securite
        action = politique_experte(pos, cible)
        if action is None:
            break
        obs_liste.append(observation(pos, cible))
        act_liste.append(action)
        pos = appliquer_action(pos, action)
    return obs_liste, act_liste

# on collecte 300 episodes de demonstration
tous_obs, toutes_actions = [], []
for episode in range(300):
    obs_ep, act_ep = generer_episode(graine=episode)
    tous_obs.extend(obs_ep)
    toutes_actions.extend(act_ep)

X = torch.tensor(np.array(tous_obs))
y = torch.tensor(toutes_actions, dtype=torch.long)

print("nombre de paires (observation, action) collectees:", X.shape[0])
print("shape d'une observation:", X.shape[1])
print("distribution des actions:", torch.bincount(y).tolist())

## 3. Behavioral cloning : entrainer un classifieur d'actions

On traite le probleme comme une classification a 4 classes (les 4 actions). C'est litteralement la meme boucle d'entrainement qu'on a vue dans le notebook PyTorch de base, seule la nature du probleme change.

In [ ]:
class PolitiqueBC(nn.Module):
    def __init__(self, dim_obs=6, dim_cachee=32, nb_actions=4):
        super().__init__()
        self.reseau = nn.Sequential(
            nn.Linear(dim_obs, dim_cachee),
            nn.ReLU(),
            nn.Linear(dim_cachee, dim_cachee),
            nn.ReLU(),
            nn.Linear(dim_cachee, nb_actions),
        )

    def forward(self, x):
        return self.reseau(x)

politique = PolitiqueBC()
fonction_loss = nn.CrossEntropyLoss()
optimizer = optim.Adam(politique.parameters(), lr=0.005)

nb_epochs = 150
for epoch in range(nb_epochs):
    optimizer.zero_grad()
    logits = politique(X)
    loss = fonction_loss(logits, y)
    loss.backward()
    optimizer.step()
    if epoch % 30 == 0:
        acc = (logits.argmax(dim=1) == y).float().mean().item()
        print(f"epoch {epoch:3d} | loss {loss.item():.4f} | accuracy sur les demos {acc:.3f}")

## 4. Evaluer la politique apprise dans le simulateur

Le vrai test n'est pas l'accuracy sur les demonstrations mais le **taux de succes** quand la politique controle elle-meme le robot, etape par etape, sur des scenarios jamais vus. C'est exactement la metrique demandee dans le Home Task officiel ("Achieves high success rate on validation and test scenarios").

In [ ]:
def executer_episode_avec_politique(politique, graine, max_pas=25):
    rng = np.random.RandomState(graine)
    pos = (rng.randint(TAILLE_GRILLE), rng.randint(TAILLE_GRILLE))
    cible = (rng.randint(TAILLE_GRILLE), rng.randint(TAILLE_GRILLE))
    for _ in range(max_pas):
        if pos == cible:
            return True
        obs = torch.tensor(observation(pos, cible)).unsqueeze(0)
        with torch.no_grad():
            action = politique(obs).argmax(dim=1).item()
        pos = appliquer_action(pos, action)
    return pos == cible

# on teste sur 200 scenarios jamais vus pendant l'entrainement (graines differentes)
nb_succes = sum(executer_episode_avec_politique(politique, graine=1000 + i) for i in range(200))
print(f"taux de succes de la politique apprise: {nb_succes / 200:.1%}")

## 5. Le piege classique du behavioral cloning : le "compounding error"

Le BC entraine le modele uniquement sur des observations **generees par l'expert**. Mais une fois deploye, le robot peut faire une petite erreur, se retrouver dans un etat que l'expert n'a jamais visite, et ne plus savoir quoi faire, ce qui cascade en encore plus d'erreurs. C'est le probleme le plus cite dans la litterature d'imitation learning (Ross & Bagnell, DAgger).

**Memotech** : "le BC copie les bonnes reponses mais n'apprend jamais a se rattraper apres une erreur".

In [ ]:
# demonstration du probleme : on force une petite perturbation aleatoire
# de temps en temps, pour voir si la politique arrive a se rattraper

def executer_episode_avec_perturbation(politique, graine, proba_perturbation=0.15, max_pas=25):
    rng = np.random.RandomState(graine)
    pos = (rng.randint(TAILLE_GRILLE), rng.randint(TAILLE_GRILLE))
    cible = (rng.randint(TAILLE_GRILLE), rng.randint(TAILLE_GRILLE))
    for _ in range(max_pas):
        if pos == cible:
            return True
        if rng.rand() < proba_perturbation:
            action = rng.randint(4)   # action aleatoire, simule une erreur ou du bruit capteur
        else:
            obs = torch.tensor(observation(pos, cible)).unsqueeze(0)
            with torch.no_grad():
                action = politique(obs).argmax(dim=1).item()
        pos = appliquer_action(pos, action)
    return pos == cible

nb_succes_perturbe = sum(executer_episode_avec_perturbation(politique, graine=1000 + i) for i in range(200))
print(f"taux de succes SANS perturbation : {nb_succes / 200:.1%}")
print(f"taux de succes AVEC 15% de perturbation aleatoire : {nb_succes_perturbe / 200:.1%}")
# la chute de performance illustre le compounding error : le modele n'a jamais vu
# ces etats "hors distribution" pendant l'entrainement et ne sait pas s'en sortir

## Piste d'amelioration : DAgger (Dataset Aggregation)

Le principe (a essayer en exercice) : au lieu de collecter toutes les demonstrations d'un coup, on alterne :
1. la politique actuelle controle le robot
2. a chaque etape, on demande a l'expert "qu'aurais-tu fait dans cet etat ?"
3. on ajoute cette paire (etat rencontre, action experte) au dataset
4. on reentraine la politique sur le dataset elargi

Ca corrige le compounding error car le modele voit desormais des etats qu'IL a lui-meme visites, pas seulement ceux de l'expert.

## Exercice (15-20 min)

1. Implemente une boucle DAgger simplifiee : genere 5 iterations, a chaque iteration fais tourner la politique actuelle sur 20 episodes, interroge `politique_experte` sur chaque etat rencontre, ajoute ces paires au dataset, reentraine
2. Compare le taux de succes final avec et sans perturbation, avant/apres DAgger
3. Bonus : passe a une grille 8x8 avec des cellules bloquees aleatoirement (comme dans le vrai Home Task) et adapte `appliquer_action` pour empecher de traverser un mur

In [ ]:
# ecris ta solution ici



